# 21.14. Проверка прогноза движения сердца по одной КТ

**Статус: протокол, расчёт не выполнен.** Это задача MP-07 из [карты MONAI Physio](README.md). Одна КТ здесь означает один трёхмерный том одной фазы сердечного цикла, а не один двумерный срез.

**Вопрос:** может ли модель, получив только опорную фазу КТ конкретного человека, предсказать изменение положения, формы и объёма сердца в остальных, действительно снятых фазах лучше простого предположения о неподвижном сердце? Полученный прогноз рассматривается как результат модели, а не как измеренное движение.

## Входы и разделение данных

1. Выбрать одно 4D-КТ-исследование с известными фазами и достаточным FOV; записать псевдоним, идентификатор исследования, единицы, систему координат, временные метки и источник масок.
2. Назначить опорную фазу **до оценки**. Для прогноза разрешены только её КТ и маска, параметры цикла и ранее обученные веса/модель формы. Остальные фазы скрыты до получения прогноза и служат проверочным набором.
3. Проверить происхождение весов и PCA-модели: использовались ли тот же человек, то же исследование или проверочные фазы при обучении, подгонке или выборе настроек. При утечке такой запуск не считается тестом прогноза.
4. Отдельно проверить совместимость анатомического определения модели (всё сердце, камеры, миокард) с нашими масками TotalSegmentator v2. Новые ручные маски здесь не предполагаются.

## Сравнения и критерии

- **Простой контроль:** та же опорная маска без движения на всех фазах. Улучшение относительно неё обязательно показывать, а не предполагать.
- **Измеренные фазы:** сравнить с масками тех же анатомических классов на реально снятых фазах; они являются автоматическим кандидатным референсом, точность которого независимо не установлена.
- **Геометрия:** для каждой фазы отдельно вычислить ошибку центра (мм), изменение объёма (мл и %), расстояния между поверхностями, перекрытие и при возможности ошибку смещений характерных точек. Показывать также ошибки *изменений относительно опорной фазы*, а не только абсолютных уровней.
- **Качество поля:** проверить направление преобразования, непрерывность, складки/отрицательный якобиан, выход за FOV и качество перенесённой сетки.
- **Временная ось:** не приравнивать автоматически метку процента R–R к механической фазе; фиксировать фактические времена КТ и правило сопоставления.
- **Перенос на других людей:** отложенные фазы одного человека проверяют временной прогноз в данном случае. Для обобщения нужен отдельный тест на человеке, исключённом из обучения и настройки.
- **Электрический смысл:** только после геометрического QC сравнить на одинаковых материалах, электродах и сеточном контроле, как прогноз и реальные фазы меняют расчётные Z и ΔZ. Это отдельный расчёт, а не метрика MONAI Physio.

## Условия выполнения и возможные исходы

Перед запуском определить доступные веса для движения сердца, их обучающий корпус, требуемую сегментацию и техническую совместимость с нашими КТ. Если готовых совместимых сердечных весов с известным происхождением нет, зафиксировать задачу прогноза как невыполнимую в режиме готового инструмента. Обучение на своих данных в текущую постановку не входит; модель лёгкого не объявлять моделью сердца.

Успех геометрического теста — заранее заданное улучшение над неподвижным контролем по выбранным метрикам при корректном QC; пороги и допустимую ошибку для FEM/ТТРКГ определить до просмотра итогов. Возможны также отсутствие выигрыша, зависимость от фазы, ошибки на отдельных камерах или непригодность данных. Любой исход сохраняется с версиями, входами и ограничениями.